# Phase 6 — Classification Head & End-to-End Inference

An interactive educational walkthrough of **Phase 5.1 (CLS Token Integration)** and **Phase 6: Classification Head** for Motor Imagery EEG Classification.
This notebook demonstrates complete forward inference from multi-band EEG tensors through Adaptive Channel Attention (ACA), Band $\times$ Channel Tokenization, 2D Positional Embeddings, CLS Token Injection, Frequency-Aware Transformer Encoder (FATE), and the MLP Classification Head (`EEGClassifier`).

## 1. Objective

Phase 6 completes the model representation architecture by converting contextualized global EEG features into unnormalized class logits for Motor Imagery (MI) tasks.

- **CLS Token Aggregation**: Instead of ad-hoc pooling across 532 tokens, a learnable `CLSToken` is prepended to the sequence. Self-attention layers dynamically aggregate global spatial-spectral information into this CLS token representation.
- **Decoupled Classification Interface**: `EEGClassifier` accepts either `FATEOutput` or raw `cls_embedding` tensors, decoupling downstream predictions from pooling choices.
- **Structured Prediction API**: `prediction = classifier(fate_out, return_metadata=True)` returns a structured `PredictionOutput` containing `logits`, `probabilities`, `predicted_class`, and `metadata`.
- **Raw Logits Output**: `ClassificationHead` returns unnormalized logits $(B, \text{num\_classes})$, leaving loss computation (e.g. CrossEntropyLoss) and optimization for Phase 7.

## 2. Theory & Mathematical Formulation

### CLS Token Injection (Phase 5.1):
Given token embeddings $E \in \mathbb{R}^{B \times N \times d_{\text{model}}}$ where $N = F \times C = 532$, a learnable CLS token $E_{\text{CLS}} \in \mathbb{R}^{1 \times 1 \times d_{\text{model}}}$ is prepended:

$$E_{\text{seq}} = [E_{\text{CLS}} \,;\, E] \in \mathbb{R}^{B \times (N+1) \times d_{\text{model}}}$$

After TransformerEncoder processing, the contextualized sequence $Z \in \mathbb{R}^{B \times (N+1) \times d_{\text{model}}}$ yields:
- Global CLS Embedding: $\mathbf{z}_{\text{CLS}} = Z[:, 0, :] \in \mathbb{R}^{B \times d_{\text{model}}}$
- Token Embeddings: $Z_{\text{tokens}} = Z[:, 1:, :] \in \mathbb{R}^{B \times N \times d_{\text{model}}}$

### Classification Head Architecture (Phase 6):
The CLS embedding $\mathbf{z}_{\text{CLS}}$ passes through a multi-layer perceptron (MLP) with Layer Normalization:

$$\mathbf{h}_1 = \text{GELU}(\text{Linear}_{d_{\text{model}} \to d_{\text{hidden}}}(\text{LayerNorm}(\mathbf{z}_{\text{CLS}})))$$
$$\mathbf{h}_2 = \text{Dropout}(p=0.3)(\mathbf{h}_1)$$
$$\mathbf{y}_{\text{logits}} = \text{Linear}_{d_{\text{hidden}} \to K}(\mathbf{h}_2) \in \mathbb{R}^{B \times K}$$

Where $K = 4$ for 4-class Motor Imagery (Left Hand, Right Hand, Both Hands, Feet).

## 3. End-to-End Architectural Flow Diagram

```text
Raw EDF Epoch (B, C, S) -> (B, 133, 250)
              │
              ▼  [Frequency Representation]
Multi-Band Tensor (B, F, C, S) -> (B, 4, 133, 250)
              │
              ▼  [Adaptive Channel Attention (ACA)]
Refined ACA Tensor (B, 4, 133, 250)
              │
              ▼  [BandChannelTokenizer: k = f*C + c]
Token Sequence (B, 532, 250)
              │
              ▼  [TemporalProjection + Band & Channel Embeddings]
Embedded Tokens (B, 532, 128)
              │
              ▼  [CLSToken Injection]
Sequence with CLS (B, 533, 128)
              │
              ▼  [FrequencyAwareTransformer (FATE)]
Transformer Output (B, 533, 128)
              │
      ┌───────┴────────────────────────┐
      ▼                                ▼
CLS Embedding (B, 128)      Contextual Tokens (B, 532, 128)
      │
      ▼  [EEGClassifier / ClassificationHead]
Raw Class Logits (B, 4) -> [Left Hand, Right Hand, Both Hands, Feet]
      │
      ▼  [Softmax (Inference Visualization)]
Class Probabilities (B, 4) & Predicted Class Labels
```

## 4. Implementation

Let's import ACA, FATE, and EEGClassifier modules and instantiate the full end-to-end model pipeline.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr and not os.path.exists(os.path.join(curr, "models")):
        parent = os.path.dirname(curr)
        if parent == curr:
            break
        curr = parent
    return curr

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"[OK] Project Root set to: {PROJECT_ROOT}")

from models.attention import ACA, AdaptiveChannelAttentionConfig
from models.transformer import FATE, FrequencyAwareTransformerConfig
from models.classifier import EEGClassifier, ClassifierConfig, PredictionOutput

print("[OK] Successfully imported ACA, FATE, and EEGClassifier modules.")

## 5. End-to-End Forward Inference Demo

Running a batch of synthetic multi-band EEG tensors `(Batch=4, Bands=4, Channels=133, Samples=250)` through the full architecture:

$$\text{Input EEG } X \xrightarrow{\text{ACA}} X_{\text{ACA}} \xrightarrow{\text{FATE}} \text{FATEOutput} \xrightarrow{\text{Classifier}} \text{PredictionOutput}$$

In [ ]:
# Load configurations from configs/model.yaml
config_path = os.path.join(PROJECT_ROOT, "configs", "model.yaml")
aca_cfg = AdaptiveChannelAttentionConfig.from_yaml(config_path)
fate_cfg = FrequencyAwareTransformerConfig.from_yaml(config_path)
clf_cfg = ClassifierConfig.from_yaml(config_path)

# Instantiate modules
aca_module = ACA(config=aca_cfg, num_channels=133, num_bands=4)
fate_module = FATE(config=fate_cfg)
clf_module = EEGClassifier(config=clf_cfg, d_model=128)

aca_module.eval()
fate_module.eval()
clf_module.eval()

# Generate sample EEG input
torch.manual_seed(42)
x_raw = torch.randn(4, 4, 133, 250)

# Execution pipeline
x_aca = aca_module(x_raw)
fate_feats, fate_out = fate_module(x_aca, return_metadata=True, band_names=["Theta", "Alpha", "Beta", "Gamma"])
prediction = clf_module(fate_out, return_metadata=True)

print("\n--- End-to-End Inference Verification ---")
print(f"Raw EEG Input Shape:        {x_raw.shape}")
print(f"ACA Refined Shape:          {x_aca.shape}")
print(f"Transformer Sequence Shape: {fate_out.contextual_embeddings.shape}")
print(f"Extracted CLS Embedding:   {fate_out.cls_embedding.shape}")
print(f"Raw Logits Shape:           {prediction.logits.shape}")
print(f"Predicted Classes (B=4):    {prediction.predicted_class.tolist()}")
print(f"Execution Time:             {prediction.metadata.execution_time_ms:.3f} ms")

## 6. Visualization & Model Inspection

### CLS Embedding Activations
Visualizing the 128-dimensional global CLS embedding representations extracted for each sample in the batch.

In [ ]:
cls_np = fate_out.cls_embedding.detach().numpy()  # (4, 128)

plt.figure(figsize=(12, 4))
plt.imshow(cls_np, aspect='auto', cmap='magma')
plt.colorbar(label="CLS Embedding Value")
plt.yticks(range(4), [f"Sample {i}" for i in range(4)])
plt.xlabel("Embedding Dimension d_model (0 to 127)")
plt.title("Global Contextual CLS Embedding Activations (4 x 128)")
plt.tight_layout()
plt.show()

### Raw Logits vs Softmax Class Probabilities
Comparing unnormalized raw logits $\mathbf{y}_{\text{logits}}$ against normalized Softmax probabilities $P(Y=k \mid X)$ across Motor Imagery classes:
1. **Class 0**: Left Hand
2. **Class 1**: Right Hand
3. **Class 2**: Both Hands
4. **Class 3**: Feet

In [ ]:
logits_np = prediction.logits.detach().numpy()
probs_np = prediction.probabilities.detach().numpy()
class_labels = ["Left Hand", "Right Hand", "Both Hands", "Feet"]

df_results = pd.DataFrame({
    "Sample": [f"Sample {i}" for i in range(4)],
    "Logit_0 (Left)": logits_np[:, 0],
    "Logit_1 (Right)": logits_np[:, 1],
    "Logit_2 (Both)": logits_np[:, 2],
    "Logit_3 (Feet)": logits_np[:, 3],
    "Prob_0 (Left)": probs_np[:, 0],
    "Prob_1 (Right)": probs_np[:, 1],
    "Prob_2 (Both)": probs_np[:, 2],
    "Prob_3 (Feet)": probs_np[:, 3],
    "Predicted Class": [f"{c} ({class_labels[c]})": c in prediction.predicted_class.tolist()]
})

display(df_results.style.format({
    "Logit_0 (Left)": "{:.3f}", "Logit_1 (Right)": "{:.3f}", "Logit_2 (Both)": "{:.3f}", "Logit_3 (Feet)": "{:.3f}",
    "Prob_0 (Left)": "{:.4f}", "Prob_1 (Right)": "{:.4f}", "Prob_2 (Both)": "{:.4f}", "Prob_3 (Feet)": "{:.4f}"
}))

### Class Probability Bar Chart for Batch Sample 0

In [ ]:
sample_probs = probs_np[0]
pred_cls = prediction.predicted_class[0].item()

plt.figure(figsize=(8, 4))
bars = plt.bar(class_labels, sample_probs, color=['cornflowerblue' if i != pred_cls else 'crimson' for i in range(4)])
plt.ylabel("Softmax Probability")
plt.ylim(0.0, 1.0)
plt.title(f"Predicted Class Probabilities for Sample 0 (Predicted: {class_labels[pred_cls]})")
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.2%}", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Conclusion & Repository Milestones

### Key Takeaways:
1. **CLS Token Integration (v0.5.1)**: Prepended learnable CLS token to sequence tokens $(N+1 = 533)$, enabling self-attention to aggregate global EEG information into $\mathbf{z}_{\text{CLS}} \in \mathbb{R}^{B \times d_{\text{model}}}$.
2. **Flexible Classifier (v0.6.0)**: `EEGClassifier` consumes `FATEOutput` directly, decoupling input pooling strategy from MLP classification logic.
3. **Structured Prediction API**: `prediction = classifier(fate_out, return_metadata=True)` returns `PredictionOutput` containing `logits`, `probabilities`, `predicted_class`, and `metadata`.
4. **Raw Logits Output**: `ClassificationHead` produces raw logits $(B, 4)$ without Softmax activation, maintaining standard PyTorch `nn.CrossEntropyLoss` compatibility.
5. **Complete Model Assembly**: The entire pipeline from raw EEG to class logits is fully modular, stateless in evaluation, and covered by 46 unit tests.

**Phase 5.1 and Phase 6 are complete and fully validated.** The architecture is now ready for **Phase 7 (Training Framework)**.